In [ ]:
from pathlib import Path

import ee
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely

from belo_horizonte.bounds import load_bounds
from belo_horizonte.temperature import get_lst
from belo_horizonte.utils import clamp_bounds

In [2]:
ee.Initialize()

In [60]:
data_path = Path("./data")
generated_path = Path("./generated")

In [4]:
bounds_ee, bounds = load_bounds(data_path=data_path, return_geometry=True)
crs = get_lst(bounds_ee).projection().crs().getInfo()

In [22]:
df_zones = (
    gpd.read_file(data_path / "PL_limite_area_verde_publica.zip")
    .drop(columns=["fid"])
    .assign(ID_AREA_VE=lambda df: df["ID_AREA_VE"].astype(int))
    .set_index("ID_AREA_VE")
    .sort_index()
    .to_crs(crs)
)

In [45]:
df_zones_joined = gpd.GeoDataFrame(
    geometry=list(df_zones.buffer(50).union_all().geoms),
    crs=crs,
).reset_index(names="zone_id")

In [58]:
def get_zone_sample_points(zone: pd.Series) -> gpd.GeoDataFrame:
    zone_bounds = clamp_bounds(*zone["geometry"].bounds, scale=30)
    zone_points = gpd.GeoDataFrame(
        geometry=[
            shapely.Point(x, y)
            for x in np.arange(zone_bounds[0], zone_bounds[2], 30)
            for y in np.arange(zone_bounds[1], zone_bounds[3], 30)
        ],
        crs=crs,
    )
    return zone_points[zone_points.intersects(zone["geometry"])].assign(
        zone_id=zone["zone_id"],
    )

In [61]:
df_sample_points = pd.concat(
    [get_zone_sample_points(zone) for _, zone in df_zones_joined.iterrows()],
    ignore_index=True,
)
df_sample_points.to_file(generated_path / "sample_points.gpkg")

In [ ]:
df_sample_points